In [ ]:
import os
from Scope.Read_Write import load_binary

In [ ]:
## In this tutorial, we will open a System file and navigate the hierarchy of classes inside.

In [ ]:
## Path of the data folder. It should be "os.path.abspath('.')+'/Data"
data_folder = os.path.abspath('.')+'/Data/'
## Loads the System object from a binary file, provided in the tutorial folder
sys = load_binary(f"{data_folder}ABITEM.npy")

In [ ]:
## All objects in SCOPE have a __repr__ method, so printing shows a summary of the object
print(sys)

In [ ]:
#The System Object mainly stores paths and sources. 

# A source is a chemical entity. There are two main types of sources: CELLs and SPECIEs:  
# -The difference between CELLs and SPECIEs is that the former is a periodic structure, so it must include cell parameters.  
# -A SPECIE is a class with various subclasses: Molecule, Ligand, and Group. Ligands and Groups only appear in Transition Metal Complexes.

# In this specific example, the system ABITEM is associated with 6 sources:
    #0: cell ABITEM01 H104-C92-N24-O4-S8-Fe4 
    #1: cell ABITEM H104-C92-N24-O4-S8-Fe4 
    #2: cell ref_hs_cell H104-C92-N24-O4-S8-Fe4 
    #3: cell ref_ls_cell H104-C92-N24-O4-S8-Fe4 
    #4: specie ref_hs_mol H18-C20-N6-S2-Fe 
    #5: specie ref_ls_mol H18-C20-N6-S2-Fe 

    # Sources 0-1 are the original cell objects imported from cell2mol
    # Sources 2-3 are the cell objects identified as reference HS and LS unit cells. This will undergo the Solid-State computations in tutorial XXX
    # Sources 4-5 are the specie objects identified as reference HS and LS molecules. This will undergo the Isolated computations in tutorial XXX

In [ ]:
# We will navigate both CELL and SPECIE Objects below.

## Sources 1: The CELL Class


In [ ]:
cell = sys.sources[0]
print(cell)

In [ ]:
# Have a look at the CELL object above. The key information is displayed, some other is accessible as attributes:

# -It has 236 atoms, whose coordinates are stored as cell.coord, and labels as cell.labels.
print("Labels and Coordinates:", cell.labels[0], cell.coord[0]) 
# -It has cell parameters, stored as cell.cell_param
print("Cell Parameters:", cell.cell_param)
# -It has 8 Molecules, stored inside cell.moleclist
print("Number of Molecules:", len(cell.moleclist))
# -It has 2 types of molecules, with formulae:
for mol in cell.refmoleclist:
    print("Unique Molecule Formula:", mol.formula)

In [ ]:
# You can also run CELL-class functions, like cell.check_fragmentation(), which checks if any molecule appears fragmented due to how the cell is cut
cell.check_fragmentation()

In [ ]:
# You can also view the unit cell with cell.view(), which opens a 3D viewer
cell.view(size='large')

## Sources 2: The SPECIE Class

In [ ]:
# Hopefully, you can identify the 4 transition metal complexes (TMCs) and the 4 solvent molecules in the viewer above.
# In any case, you can access those molecules through the cell.moleclist attribute. Taking the first molecule as an example:
mol = cell.moleclist[0]
# Again, the key information is displayed. The rest is accessible as attributes
print(mol)


In [ ]:
# Notice the molecule is of type=SPECIE, and subtype=MOLECULE. Molecules are subclasses of SPECIEs. 
# Also, this molecule is a TMC, as shown by:
print(mol.iscomplex)
# As a result, it has a ligand list (mol.ligands), which contains 3 Ligand objects:
print(len(mol.ligands), "ligands in this TMC")
# And a list of metal atoms:
print(len(mol.metals), "metal atom(s) in this TMC")

In [ ]:
## Apart from attributes, species -and thus also molecules- also have useful functions, e.g.:
print(mol.get_centroid())

In [ ]:
## All species can be visualized with the .view() method:
mol.view()

In [ ]:
## You can access the metal and ligand objects as:
met = mol.metals[0]
print(met)
lig = mol.ligands[0]
print(lig)

In [ ]:
## The METAL is a subclass of ATOM (type=atom, subtype=metal). Apart from attributes, it also has useful functions, e.g.:
met.get_cshm()   ## This one retrieves the Coordination Shape Measure of the metal.
                 ## The CShM quantifies how close the coordination environment is to an ideal geometry, which is an octahedron by default.

In [ ]:
## The LIGAND is also a subclass of SPECIE (type=specie, subtype=ligand). It has some useful functions too, e.g.:
print(lig.get_denticity())
## This one retreives the atom object of all atoms that are connected to a metal, in this case only an N atom:
print(lig.get_connected_atoms())

In [ ]:
## So far, we went down in the hierarchy: System -> Cell -> Molecule -> Metal/Ligand. You can also go back up:
parent = lig.get_parent('cell') # This retrieves the parent CELL object of the ligand
print(parent)

In [ ]:
idx = lig.get_parent_indices('molecule') # This retrieves the index of the ligand atoms in the parent molecule
print(idx)

In [ ]:
## The cell in the example were imported from cell2mol CELL objects. Ligands in these cells are associated with an rdkit object:
#lig.rdkit_obj  ## Uncomment to visualize it

## And a Smiles
lig.smiles

In [ ]:
## In transition metal complexes, molecule smiles are the ligand smiles concatenated. New better ways have been proposed, but not yet implemented in SCOPE.
mol.smiles

## Also, the ATOM and BOND Classes

In [ ]:
# As we've briefly seen above, species have atoms. They also have bonds. Both are classes in SCOPE.
len(mol.atoms)

In [ ]:
## The first atom of the molecule happens to be the metal. The second is a regular atom. Notice the different subtype
for at in mol.atoms[0:10]:                                  ## Printing the first 10 atoms
    print(at.get_parent_index('molecule'), at.label, at.subtype, at.charge, at.madjnum, at.adjnum)

## at.charge stores the formal atomic charge.
## at.madjnum is the number of adjacencies the atom has with a metal center.
## at.adjnum is the total number of adjacencies the atom has with any other atom

In [ ]:
## Bonds are also stored as objects, with very limited functionality for now:
at = mol.atoms[1]
print(at.bonds)

In [ ]:
## The formal bond order (bond.order) is extracted from the rdkit object, if available: